In [ ]:
import pandas as pd

movies = pd.read_csv("movies.csv")

In [ ]:
movies


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
62418,209157,We (2018),Drama
62419,209159,Window of the Soul (2001),Documentary
62420,209163,Bad Poems (2018),Comedy|Drama
62421,209169,A Girl Thing (2001),(no genres listed)


In [ ]:
import re

def clean_title(title):
    return re.sub("[^a-zA-Z0-9 ]","", title)

In [ ]:
movies["clean_title"]= movies["title"].apply(clean_title)

In [ ]:
movies


,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II 1995
...,...,...,...,...
62418,209157,We (2018),Drama,We 2018
62419,209159,Window of the Soul (2001),Documentary,Window of the Soul 2001
62420,209163,Bad Poems (2018),Comedy|Drama,Bad Poems 2018
62421,209169,A Girl Thing (2001),(no genres listed),A Girl Thing 2001


In [ ]:
#convert our titles to a set of numbers so that our computer can understand it
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1,2))

tfidf = vectorizer.fit_transform(movies["clean_title"])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search(title):
  title = clean_title(title)
  query_vec = vectorizer.transform([title])
  similarity = cosine_similarity(query_vec, tfidf).flatten()
  indices =np.argpartition(similarity,-5)[-5:]
  results = movies.iloc[indices][::-1]
  return results

In [ ]:
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
    value="Toy Story",
    description="Movie Title:",
    disabled=False
)
movie_list =  widgets.Output()

def on_type(data):
    with movie_list:
        movie_list.clear_output()
        title = data["new"]
        if len(title) > 5:
            display(search(title))
movie_input.observe(on_type, names="value")

display(movie_input, movie_list)


Text(value='Toy Story', description='Movie Title:')

Output()

In [ ]:
results

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 2 1999
59767,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,Toy Story 4 2019
14813,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,Toy Story 3 2010
20497,106022,Toy Story of Terror (2013),Animation|Children|Comedy,Toy Story of Terror 2013


In [ ]:
ratings = pd.read_csv("ratings.csv")

In [ ]:
ratings

,userId,movieId,rating,timestamp
0,1,296.0,5.0,1.147880e+09
1,1,306.0,3.5,1.147869e+09
2,1,307.0,5.0,1.147869e+09
3,1,665.0,5.0,1.147879e+09
4,1,899.0,3.5,1.147869e+09
...,...,...,...,...
85373,647,9010.0,2.5,1.330432e+09
85374,647,27402.0,4.0,1.506807e+09
85375,647,27660.0,3.0,1.456428e+09
85376,647,27904.0,3.5,1.509057e+09


In [ ]:
ratings.dtypes

userId         int64
movieId      float64
rating       float64
timestamp    float64
dtype: object

In [ ]:
movie_id = 1

In [ ]:
similar_users = ratings[(ratings["movieId"] == movie_id) & (ratings["rating"] > 4)]["userId"].unique()

In [ ]:
similar_users

array([ 36,  75,  86,  90,  93,  95,  96,  98, 111, 120, 127, 143, 152,
       158, 160, 162, 171, 186, 188, 211, 217, 229, 230, 235, 249, 257,
       259, 297, 298, 302, 323, 329, 355, 359, 369, 371, 381, 392, 402,
       411, 428, 435, 439, 447, 449, 468, 469, 477, 484, 513, 519, 537,
       540, 541, 548, 551, 553, 561, 567, 573, 582, 593, 607, 609, 611,
       623, 624, 626, 628, 631, 644])

In [ ]:
similar_users_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"]>4)]["movieId"]

In [ ]:
similar_users_recs

5101       1.0
5105      34.0
5111     110.0
5114     150.0
5127     260.0
         ...  
85171    356.0
85173    380.0
85182    588.0
85183    589.0
85186    593.0
Name: movieId, Length: 4526, dtype: float64

In [ ]:
similar_users_recs = similar_users_recs.value_counts() / len(similar_users)

similar_users_recs = similar_users_recs[similar_users_recs>.1]

In [ ]:
similar_users_recs

movieId
1.0       1.000000
318.0     0.492958
593.0     0.352113
356.0     0.323944
296.0     0.323944
            ...   
1527.0    0.112676
1222.0    0.112676
2692.0    0.112676
4963.0    0.112676
1090.0    0.112676
Name: count, Length: 93, dtype: float64

In [ ]:
all_users = ratings[(ratings["movieId"].isin(similar_users_recs.index)) & (ratings["rating"] > 4)]

In [ ]:
all_users_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

In [ ]:
all_users_recs

movieId
318.0      0.333333
296.0      0.258333
527.0      0.246667
593.0      0.230000
2571.0     0.226667
             ...   
2692.0     0.045000
2542.0     0.040000
3114.0     0.040000
1090.0     0.036667
78499.0    0.026667
Name: count, Length: 93, dtype: float64

In [ ]:
rec_percentages = pd.concat([similar_users_recs, all_users_recs], axis=1)
rec_percentages.columns = ["similar", "all"]


In [ ]:
rec_percentages

,similar,all
movieId,,
1.0,1.000000,0.118333
318.0,0.492958,0.333333
593.0,0.352113,0.230000
356.0,0.323944,0.223333
296.0,0.323944,0.258333
...,...,...
1527.0,0.112676,0.060000
1222.0,0.112676,0.056667
2692.0,0.112676,0.045000


In [ ]:
rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]
rec_percentages = rec_percentages.sort_values("score", ascending=False)

In [ ]:
rec_percentages

,similar,all,score
movieId,,,
1.0,1.000000,0.118333,8.450704
3114.0,0.197183,0.040000,4.929577
78499.0,0.126761,0.026667,4.753521
4886.0,0.267606,0.068333,3.916180
1073.0,0.154930,0.046667,3.319920
...,...,...,...
2959.0,0.239437,0.195000,1.227880
1196.0,0.211268,0.175000,1.207243
527.0,0.295775,0.246667,1.199086


In [ ]:
rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")

,similar,all,score,movieId,title,genres,clean_title
0,1.000000,0.118333,8.450704,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
3021,0.197183,0.040000,4.929577,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 2 1999
14813,0.126761,0.026667,4.753521,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,Toy Story 3 2010
4780,0.267606,0.068333,3.916180,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,Monsters Inc 2001
1047,0.154930,0.046667,3.319920,1073,Willy Wonka & the Chocolate Factory (1971),Children|Comedy|Fantasy|Musical,Willy Wonka the Chocolate Factory 1971
2025,0.169014,0.051667,3.271240,2115,Indiana Jones and the Temple of Doom (1984),Action|Adventure|Fantasy,Indiana Jones and the Temple of Doom 1984
6258,0.197183,0.061667,3.197564,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,Finding Nemo 2003
580,0.225352,0.073333,3.072983,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,Aladdin 1992
8246,0.225352,0.073333,3.072983,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,Incredibles The 2004
1063,0.112676,0.036667,3.072983,1090,Platoon (1986),Drama|War,Platoon 1986


In [ ]:
def find_similar_movies(movie_id):
  similar_users = ratings[(ratings["movieId"] == movie_id) & (ratings["rating"] > 4)]["userId"].unique()
  similar_users_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"]>4)]["movieId"]

  similar_users_recs = similar_users_recs.value_counts() / len(similar_users)
  similar_users_recs = similar_users_recs[similar_users_recs>.10]

  all_users = ratings[(ratings["movieId"].isin(similar_users_recs.index)) & (ratings["rating"] > 4)]
  all_users_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

  rec_percentages = pd.concat([similar_users_recs, all_users_recs], axis=1)
  rec_percentages.columns = ["similar", "all"]

  rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]

  rec_percentages = rec_percentages.sort_values("score", ascending=False)
  return rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")




In [ ]:
movie_input_name = widgets.Text(
    value="Toy Story",
    description="Movie Title:",
    disabled=False
)

recommendation_list =  widgets.Output()

def on_type(data):
  with recommendation_list:
    recommendation_list.clear_output()
    title = data["new"]
    if len(title) > 5:
      results = search(title)
      movie_id = results.iloc[0]["movieId"]
      display(find_similar_movies(movie_id))

movie_input_name.observe(on_type, names="value")

display(movie_input_name, recommendation_list)

Text(value='Toy Story', description='Movie Title:')

Output()